# Phase 3: LoRA Fine-tuning with NeMo
## Nemotron Reasoning Challenge

In [ ]:
# First, check available libraries
import subprocess
result = subprocess.run(['pip', 'list'], capture_output=True, text=True)
print("Checking for relevant packages...")
for line in result.stdout.split('\n'):
    line_lower = line.lower()
    if any(x in line_lower for x in ['nemo', 'unsloth', 'transformers', 'trl', 'peft', 'axolotl', 'torch']):
        print(line)

## 3.1 Setup Environment

In [ ]:
# Check if we need to install additional packages
try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    print("Installing transformers...")
    !pip install transformers

In [ ]:
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
except ImportError:
    print("Installing PEFT...")
    !pip install peft

In [ ]:
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not available")

## 3.2 Load and Prepare Data

In [ ]:
import pandas as pd
import json

# Load data
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

In [ ]:
# Create formatted training data
def create_training_example(prompt, answer):
    return {
        'text': f"""Instruction: Solve the puzzle step by step.

{prompt}

Answer: {answer}<|end_of_text|>"""
    }

train_data = []
for _, row in train_df.iterrows():
    train_data.append(create_training_example(row['prompt'], row['answer']))

print(f"Created {len(train_data)} training examples")
print("\nSample:")
print(train_data[0]['text'][:300])

## 3.3 Model Configuration

In [ ]:
# Model configuration
MODEL_NAME = "nvidia/Nemotron-3-Nano-30B"
OUTPUT_DIR = "./lora_adapter"

print(f"Target model: {MODEL_NAME}")
print(f"Output directory: {OUTPUT_DIR}")

## 3.4 LoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")

## 3.5 Training Function (for when GPU is available)

In [ ]:
# This is a template for training - requires sufficient GPU memory
"""
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

def train_lora_adapter(model_name, train_data, output_dir):
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Add LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Prepare dataset
    dataset = Dataset.from_list(train_data)
    
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(examples['text'], truncation=True, max_length=2048)
    
    dataset = dataset.map(tokenize_function, batched=True)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
    )
    
    # Train
    trainer.train()
    
    # Save
    model.save_pretrained(output_dir)
    
    return model, tokenizer
"""

print("Training function defined (commented out - requires GPU)")

## 3.6 Alternative: Create LoRA with QLoRA (4-bit quantization)

In [ ]:
# For limited GPU memory, we can use QLoRA with 4-bit quantization
"""
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
"""

print("QLoRA configuration available (commented out - requires GPU)")

## 3.7 Generate Submission Template

In [ ]:
# For now, let's create a baseline submission using rule-based solving where possible
# This will be replaced with model predictions after fine-tuning

import re

def extract_target_input(prompt):
    """Extract the target input from the prompt."""
    # For bit manipulation: find the target binary string
    match = re.search(r'Now, determine the output for:\s*([01]{8})', prompt)
    if match:
        return match.group(1), 'bit'
    
    # For text decryption: find the target text
    match = re.search(r'Now, decrypt the following text:\s*(.+)', prompt)
    if match:
        return match.group(1).strip(), 'text'
    
    # For numeral: find the number
    match = re.search(r'write the number (\d+) in the Wonderland', prompt)
    if match:
        return match.group(1), 'numeral'
    
    return None, 'unknown'

In [ ]:
# Test extraction on test data
for i, row in test_df.iterrows():
    target, ptype = extract_target_input(row['prompt'])
    print(f"Test {i+1}: type={ptype}, target={target}")

## Summary
- Set up LoRA configuration with PEFT
- Defined training functions for full precision and QLoRA
- Created extraction helpers for test data
- Next: Run actual training when GPU is available, then generate predictions